In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()

#Download dataset
train_dataset = datasets.MNIST(
    root="../data", #the 
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

#Make batches with size of 64
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [2]:
cnn = nn.Sequential(

    nn.Conv2d( # Make a Kernel 
        in_channels=1, # inputs 1 image
        out_channels=16, # has 16 different kernel filters and outputs 16 images
        kernel_size=3, # makes a 3x3 filter
        padding=1 # adds a boarder of 0s so the kernel outputs the same x and y dimensions [28][28]
    ),
    nn.ReLU(), # makes all negative numbers 0
    nn.MaxPool2d(kernel_size=2), #takes the largest in a 2x2 making the model have 1/4th of the amount of pixels so its a [14][14]


    
    nn.Conv2d( # Make a Kernel
        in_channels=16, # inputs 16 images
        out_channels=32, # for each of these 16 images outputs 32 images so in total 512 images
        kernel_size=3, # the filter is a 3x3
        padding=1 # adds a boarder of 0s so the kernel outputs the same x and y dimensions [14][14]
    ),
    nn.ReLU(), # makes all negative numbers 0
    nn.MaxPool2d(kernel_size=2), #takes the largest in a 2x2 making the model have 4 time less pixels so its a [7][7]


    
    # Classification of the picture to numbers
    nn.Flatten(), # makes the torch tensor of [64, 32, 7, 7] into a [64, 1568] - (64 because of batch size being 64)
    nn.Linear(32 * 7 * 7, 10) # places the [64, 1568] into a linear model with an input of 64 and an outpur of 10 (the 10 numbers)
).to(device)


#Create loss and optimizer
cnn_loss_fn = nn.CrossEntropyLoss()

cnn_optimizer = torch.optim.Adam(
    cnn.parameters(),
    lr=0.001
)

In [3]:
#stores the losses for each epoch
cnn_train_loss = []

for epoch in range(5):
    cnn.train()
    epoch_loss = [] # stores the loss and resets for each epoch

    for images, labels in train_loader: # train loader is split up by 64, Loops over the 64 batches and assigns the images to
                                        # a tensor[64][1][28][28] an
        images = images.to(device)
        labels = labels.to(device)

        cnn_optimizer.zero_grad()

        output = cnn(images)
        loss = cnn_loss_fn(output, labels)

        epoch_loss.append(loss.item())

        loss.backward()
        cnn_optimizer.step()

    average_loss = sum(epoch_loss) / len(epoch_loss)
    cnn_train_loss.append(average_loss)

    print(f"Epoch {epoch + 1} -- Average Loss: {average_loss:.4f}")

Epoch 1 -- Average Loss: 0.2525
Epoch 2 -- Average Loss: 0.0719
Epoch 3 -- Average Loss: 0.0523
Epoch 4 -- Average Loss: 0.0430
Epoch 5 -- Average Loss: 0.0361


In [4]:
cnn.eval()

correct_predictions = 0
total_predictions = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        output = cnn(images)
        prediction = torch.argmax(output, dim=1)

        correct_predictions += (prediction == labels).sum().item()
        total_predictions += len(labels)

cnn_test_accuracy = correct_predictions / total_predictions

print(f"CNN test accuracy: {cnn_test_accuracy:.2%}")

CNN test accuracy: 98.60%
